In [ ]:
!pip install -r requirements.txt
from warpdrive import WarpDrive
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pickle
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import LabelEncoder
from warpdrive import WarpDrive
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from catboost import CatBoostClassifier
import pickle
import h2o
h2o.init()
wd = WarpDrive()
df = wd.get_args("df")
col = wd.get_args("col")
mod_sc = wd.get_args("mod_sc")
mod_ts = wd.get_args("mod_ts")
cf_html = wd.get_args("cf_html")
cf_png = wd.get_args("cf_png")
username = wd.get_args("username")
password = wd.get_args("password")
port = wd.get_args("port")
host = wd.get_args("host")
df_up = wd.get_args("df_up")
mod_ext = wd.get_args("mod_ext")
mod_ext_stat = wd.get_args("mod_ext_stat")
mod_ext_skl = wd.get_args("mod_ext_skl")
# your code here

wd.save_table(df)
print(mod_sc)
print(mod_ts)
print(mod_ext)
print(mod_ext_stat)
print(mod_ext_skl)
wd.create_df(col)
wd.save_html(cf_html)
wd.save_png(cf_png)
# wd.save_graph(cf_jpg)
#df1 = pd.read_csv('consolefiles/cf_csv.csv')
wd.save_table(col)
print("TEST")
x = [1, 2, 3, 4, 5]
y = [2, 3, 5, 7, 11]
#Plot
plt.plot(x, y, marker='o', linestyle='-')
# Add labels and title
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.title('Simple Line Plot')
# Show grid
plt.grid(True)
wd.save_image(plt)
# Show plot
categories = ['A', 'B', 'C', 'D']
values = [10, 20, 15, 25]
# Create a bar trace
trace = go.Bar(x=categories, y=values)
# Create the layout
layout = go.Layout(title='Bar Chart Example', xaxis=dict(title='Categories'), yaxis=dict(title='Values'))
# Create the figure
fig = go.Figure(data=[trace], layout=layout)
wd.save_graph(fig)
# Data
categories = ['A', 'B', 'C', 'D']
values = [10, 20, 15, 25]
# Create a bar trace with custom colors
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']  # Custom color palette
trace = go.Bar(x=categories, y=values, marker=dict(color=colors), text=values, textposition='outside')
# Create the layout with improved styling
layout = go.Layout(
    title=dict(text='Enhanced Bar Chart', x=0.5, font=dict(size=24)),  # Centered title
    xaxis=dict(title='Categories', tickfont=dict(size=14)),
    yaxis=dict(title='Values', tickfont=dict(size=14)),
    template='plotly_white',  # Clean background
    margin=dict(t=80, b=50, l=50, r=50),  # Adjust margins
)
# Create the figure
fig = go.Figure(data=[trace], layout=layout)
# wd.save_graph(fig)
# Save the graph as an HTML file
fig.write_html("enhanced_bar_chart.html")
# Display the graph
#fig.show()
wd.save_graph(fig)
# Display the figure


X = df[['Loan_Amount', 'Home_Owner']]  # Features
y = df['Gender']  # Target
# Step 3: Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Step 4: Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)
# Step 5: Evaluate the model (optional)
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
 
exog_columns = ["Loan_Amount", "Home_Owner"]
wd.create_model(
    model=model,
    library="sklearn",
    model_technique="LogisticRegressionClassifier",
    input_variables=exog_columns,  # Only exogenous columns are used
    target_column="Gender",
    train_table="df",
    lags=0,
    exog_columns=exog_columns,
)
# Step 6: Save the model to a .pkl file
with open('logistic_model.pkl', 'wb') as file:
   pickle.dump(model, file)
 
# Train the CatBoostClassifier only on the exogenous columns
clf = CatBoostClassifier(random_state=0).fit(df[exog_columns].values, df['Gender'].values)
# Update col to reflect that only exog_columns are used
col = exog_columns
# Create the model using the specified exogenous columns
wd.create_model(
    model=clf,
    library="catboost",
    model_technique="CatboostClassifier",
    input_variables=col,  # Only exogenous columns are used
    target_column="Gender",
    train_table="df",
    lags=0,
    exog_columns=exog_columns,
)
with open('cbc_model.pkl', 'wb') as file:
    pickle.dump(clf, file)

 
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
 
 
modelts = SARIMAX(y, exog=X, order=(1, 1, 1), seasonal_order=(0, 0, 0, 0))
results = modelts.fit(disp=False)
 
# Step 5: Print model summary safely
try:
    summary = results.summary()
    print(summary)
except Exception as e:
    summary = None
 
print("ARIMA model saved as 'arima_model.pkl'")
 
wd.create_model(
    model=results,
    library="statsmodels",
    model_technique="ARIMA",
    input_variables=exog_columns,
    time_column="Date",
    target_column="Gender",
    train_table="df",
    lags=0,
    exog_columns=exog_columns,
)
 
with open('arima_model.pkl', 'wb') as file:
    pickle.dump(results, file)

 
df.index = range(len(df))
df_up['Age_copy'] = df_up['Age']
wd.update_df(df_up, "df_up")
wd.create_df(df, "UDF_UPDATED_DF")

wd.create_sas_connection(
    username=username,
    password=password,
    host=[host],
    port=port
)